To train this agent, click _Runtime_ and press _Run all_. Make sure you've enabled a free Tesla T4 GPU!

<div class="align-center">
<a href="https://github.com/openpipe/art"><img src="https://github.com/openpipe/art/raw/main/assets/ART_pill.png" height="50"></a>
<a href="https://discord.gg/zbBHRUpwf4"><img src="https://github.com/openpipe/art/raw/main/assets/Discord_pill.png" height="50"></a>
<a href="https://openpipe.ai/blog/art-e-mail-agent"><img src="https://github.com/openpipe/art/raw/main/assets/ART_E_pill.png" height="50"></a>

Questions? Join the Discord and ask away! For feature requests or to leave a star, visit our [Github](https://github.com/openpipe/art).

</div>

<a href="https://art.openpipe.ai/"><img src="https://github.com/openpipe/art/raw/main/assets/Header_separator.png" height="5"></a>

This notebook shows how to train a Qwen 2.5 3B model to play tic tac toe. It will demonstrate how to set up a multi-turn agent, how to train it, and how to evaluate it.

Completions will be logged to OpenPipe, and metrics will be logged to Weights & Biases.

You will learn how to construct an [agentic environment](#Environment), how to define a [rollout](#Rollout), and how to run a [training loop](#Loop).


### WARNING:

If you are running in Google Colab and installing numpy does not say "Requirement already satisfied: numpy<2.0.0" then click "Runtime" and "Restart Session."


In [ ]:
# make sure we're using numpy 1.*.*
import numpy as np

if (np.__version__).startswith("1."):
    print("Numpy version is 1.*.*, you're good to go!")
else:
    raise ValueError("Please restart your runtime using the above instructions!")

### Environment Variables

Later on in the notebook, we'll be creating a model that can automatically logs metrics to Weights & Biases. In order to do so, you'll need to provide your Weights & Biases API key as an environment variable.

You can also optionally initiate an OpenPipe client to report completions to a [dashboard](https://app.openpipe.ai) to get a feel for what the completions your model is generating look like, and how they change over time. Logging to OpenPipe is free, but is not required for training!


In [ ]:
import os


# Optional
WANDB_API_KEY = ""
if WANDB_API_KEY:
    os.environ["WANDB_API_KEY"] = WANDB_API_KEY

# Optional
OPENPIPE_API_KEY = "opk_90bbdee21c12388777592e16fe2b570014e64c7f18"
OPENPIPE_API_KEY = ""
if OPENPIPE_API_KEY:
    os.environ["OPENPIPE_API_KEY"] = OPENPIPE_API_KEY

### Agentic Environment

<a name="Environment"></a>

ART allows your agent to learn by interacting with its environment. In this example, we'll create an environment in which the agent can play tic tac toe.

Feel free to read as much or as little of this section's code as you'd like. The important thing to understand is that we're defining the rules of this agent's environment. In many cases, this will already be defined by the task you're trying to solve, but if you need to define a custom environment, this is how you do it.


In [ ]:
import random
from typing import TypedDict
from typing import Literal
import xml.etree.ElementTree as ET
import re

class TicTacToeGame(TypedDict):
    board: list[list[str]]
    agent_symbol: Literal["x", "o"]
    opponent_symbol: Literal["x", "o"]

def generate_game(board_length: int = 3) -> TicTacToeGame:
    board = [["_" for _ in range(board_length)] for _ in range(board_length)]
    agent_symbol = random.choice(["x", "o"])
    opponent_symbol = "x" if agent_symbol == "o" else "o"
    return {
        "board": board,
        "agent_symbol": agent_symbol,
        "opponent_symbol": opponent_symbol,
    }


def render_board(game: TicTacToeGame) -> str:
    board = game["board"]
    board_length = len(board)
    # print something like this:
    #    1   2   3
    # A  _ | x | x
    # B  o | _ | _
    # C  _ | o | _
    # where _ is an empty cell

    board_str = "   " + "   ".join([str(i + 1) for i in range(board_length)]) + "\n"
    for i in range(board_length):
        board_str += f"{chr(65 + i)}  {board[i][0]} | {board[i][1]} | {board[i][2]}\n"
    return board_str


def get_opponent_move(game: TicTacToeGame) -> tuple[int, int]:
    # get a random empty cell
    empty_cells = [
        (i, j) for i in range(3) for j in range(3) if game["board"][i][j] == "_"
    ]
    return random.choice(empty_cells)


def apply_agent_move(game: TicTacToeGame, move: str) -> None:
    board_length = len(game["board"])

    try:
        root = ET.fromstring(move)
        square = root.text
    except Exception:
        raise ValueError("Invalid xml")

    try:
        row_index = ord(square[0]) - 65
        col_index = int(square[1]) - 1
    except Exception as e:
        print(e)
        raise ValueError("Unable to parse square")

    if (
        row_index < 0
        or row_index >= board_length
        or col_index < 0
        or col_index >= board_length
    ):
        raise ValueError(
            f"Invalid move, row or column out of bounds: {row_index}, {col_index}"
        )

    # check if the move is valid
    if game["board"][row_index][col_index] != "_":
        raise ValueError("Square already occupied")

    game["board"][row_index][col_index] = game["agent_symbol"]


def check_winner(board: list[list[str]]) -> Literal["x", "o", "draw", None]:
    board_length = len(board)
    # check rows
    for row in board:
        if row.count(row[0]) == board_length and row[0] != "_":
            return row[0]
    # check columns
    for col in range(board_length):
        if [board[row][col] for row in range(board_length)].count(
            board[0][col]
        ) == board_length and board[0][col] != "_":
            return board[0][col]

    # top right to bottom left
    upward_diagonal = [board[i][board_length - i - 1] for i in range(board_length)]
    if (
        upward_diagonal.count(upward_diagonal[0]) == board_length
        and upward_diagonal[0] != "_"
    ):
        return upward_diagonal[0]

    # top left to bottom right
    downward_diagonal = [board[i][i] for i in range(board_length)]
    if (
        downward_diagonal.count(downward_diagonal[0]) == board_length
        and downward_diagonal[0] != "_"
    ):
        return downward_diagonal[0]

    # check for draw
    if all(cell != "_" for row in board for cell in row):
        return "draw"
    return None


### Creating a Model

Now that we've defined the rules of our environment, we can create a model that will learn to play 2048. We'll use a Qwen 2.5 3B model for this example. The `name` parameter will be associated with a wandb run, and the `base_model` parameter is the model that we'll be training a LoRA on top of.

In [ ]:
import art
from dotenv import load_dotenv

from openpipe.client import OpenPipe
from art.local import LocalBackend

load_dotenv()

op_client = OpenPipe()
print("OpenPipe client initialized")

random.seed(42)

backend = LocalBackend(path="./.art")

### Creating a Model

Now that we've defined the rules of our environment, we can create a model that will learn to play tic tac toe. We'll use a Qwen 2.5 3B model for this example. The `name` parameter will be associated with a wandb run, and the `base_model` parameter is the model that we'll be training a LoRA on top of.


In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

model_name = "Qwen/Qwen2.5-7B-Instruct"
project_name = "tic-tac-toe-dac_agent"
run_name = "003"

model = art.TrainableModel(
    name=run_name, project=project_name, base_model=model_name,
)
await model.register(backend)

### Defining a Rollout

<a name="Rollout"></a>

A rollout is a single episode of an agent performing its task. It generates one or more trajectories, which are lists of messages and choices.

In this example, the rollout function generates a game of tic tac toe, and the agent plays it until the game is finished. It then returns a trajectory which contains all the `system` and `user` messages presented to the agent, as well as all the `choices` that the agent made.

When the game is finished the `reward` for the agent's performance is calculated based on whether the agent won, lost, drew, or errored, which is then assigned to the trajectory.

This rollout function will be called many times in parallel during each step of the training loop.

In [ ]:
import re

def extract_text_between_markers(
    text: str, start_marker: str, end_marker: str
) -> list[str]:
    """
    Extracts all instances of text between two specific markers in a string.

    Args:
        text: The input string to parse.
        start_marker: The beginning marker.
        end_marker: The ending marker.

    Returns:
        A list of strings, where each string is an instance of text found between the markers.
    """
    # Create a regex pattern to find text non-greedily between the markers
    # re.escape is used to escape any special characters in the markers
    # (.*?) matches any character (except newline) zero or more times, non-greedily
    pattern = re.escape(start_marker) + r"(.*?)" + re.escape(end_marker)

    # Find all non-overlapping matches of the pattern in the string
    matches = re.findall(pattern, text, re.DOTALL)

    return matches

In [ ]:
import art
import openai
import time
import math
from pydantic import BaseModel
from openai.types.chat import ChatCompletionDeveloperMessageParam, ChatCompletionToolMessageParam
import os
from openai import AsyncOpenAI
from typing import defaultdict
from art.openai import patch_openai
from sys_prompt import SystemPrompt, DaCSystemPrompt
from dac_agent import DACAgent

timings = dict(
    total_chat_time=[],
    total_rollout_time=[],
)

class TicTacToeScenario(BaseModel):
    step: int

@art.retry(exceptions=(openai.LengthFinishReasonError,))
async def rollout(
    model: art.Model, scenario: TicTacToeScenario, model_inference_name, client: AsyncOpenAI,
) -> art.Trajectory:
    total_chat_time = 0
    total_rollout_time = time.time()

    agent = DACAgent(
        client=client,
        model=model_inference_name,
        model_system_message=SystemPrompt.Qwen,
        dac_sys_prompt= DaCSystemPrompt.dac_sys_prompt_v2,
        max_depth=2,
    )

    game_message = {
        "role": "user",
        "content": "You are a tic-tac-toe player. You are playing against an opponent. Always choose the move most likely to lead to an eventual win. Return your move as an XML object with a single property 'move', like so: <move>A1</move>. Optional moves are 'A1', 'B3', 'C2', etc. You are the {game['agent_symbol']} symbol. Give short and concise answers."
    }
    agent.trajectory.messages_and_choices.append(game_message)

    game = generate_game()
    move_number = 0

    if game["agent_symbol"] == "o":
        starting_opponent_move = get_opponent_move(game)
        game["board"][starting_opponent_move[0]][starting_opponent_move[1]] = game[
            "opponent_symbol"
        ]

    while check_winner(game["board"]) is None:
        current_board = render_board(game)
        current_board_msg = {
            "role": "user",
            "content": current_board,
        }
        try:
            requested_at = time.time()
            trajectory = await agent.chat(current_board_msg)
            total_chat_time += time.time() - requested_at
        except openai.LengthFinishReasonError as e:
            raise e
        except Exception as e:
            print("caught exception generating chat completion")
            print(e)
            global failing_trajectory
            failing_trajectory = trajectory
            raise e

        # try:
        #     if op_client.api_key:
        #         op_client.report(
        #             requested_at=requested_at,
        #             received_at=int(time.time() * 1000),
        #             req_payload={
        #                 "model": model.name,
        #                 "messages": messages,
        #                 "metadata": {
        #                     "notebook-id": "tic-tac-toe",
        #                     "step": str(scenario.step),
        #                     "move_number": str(move_number),
        #                 },
        #             },
        #             resp_payload=chat_completion,
        #             status_code=200,
        #         )
        # except Exception as e:
        #     print(f"Error reporting to OpenPipe: {e}")

        content = trajectory.messages()[-1]["content"]
        move =  extract_text_between_markers(
            content, "<move>", "</move>"
        )

        try:
            apply_agent_move(game, "<move>" + move[-1] + "</move>")
        except ValueError:
            trajectory.reward = -1 + (math.log(move_number + 1) / math.log(100))
            break
        except IndexError:
            trajectory.reward = -1 + (math.log(move_number + 1) / math.log(100))
            break

        move_number += 1
        if check_winner(game["board"]) is not None:
            break

        opponent_move = get_opponent_move(game)
        game["board"][opponent_move[0]][opponent_move[1]] = game["opponent_symbol"]

    winner = check_winner(game["board"])

    if winner == game["agent_symbol"]:
        trajectory.reward = 1
        trajectory.metrics["win"] = 1
    elif winner == game["opponent_symbol"]:
        trajectory.reward = 0
        trajectory.metrics["win"] = 0
    elif winner == "draw":
        trajectory.reward = 0.5
        trajectory.metrics["win"] = 0.5

    trajectory.metrics["num_moves"] = move_number

    total_rollout_time = time.time() - total_rollout_time
    # if op_client.api_key:
    #     try:
    #         reported_win = (
    #             trajectory.metrics["win"] if "win" in trajectory.metrics else -1
    #         )
    #         op_client.report(
    #             requested_at=requested_at,
    #             received_at=int(time.time() * 1000),
    #             req_payload={
    #                 "model": model.name,
    #                 "messages": messages,
    #                 "metadata": {
    #                     "notebook-id": "tic-tac-toe",
    #                     "step": str(scenario.step),
    #                     "num_moves": str(move_number),
    #                     "win": str(reported_win),
    #                     "reward": str(trajectory.reward),
    #                     "choice_id": choice.index,
    #                 },
    #             },
    #             resp_payload=chat_completion,
    #             status_code=200,
    #         )
    #     except Exception as e:
    #         print(f"Error reporting to OpenPipe: {e}")

    timings["total_chat_time"].append(total_chat_time)
    timings["total_rollout_time"].append(total_rollout_time)
    return trajectory


<a name="Loop"></a>

### Training Loop

The training loop is where the magic happens. For each of the 100 steps defined below, the rollout function will be called 200 times in parallel. This means that 200 games will be played at once. Each game will produce a trajectory, which will be used to update the model.

The `gather` step will wait for all of the trajectories to be generated, then it will delete all but the most recent checkpoint and train the model on the new trajectories.

Inference will be blocked until the training is complete.


In [ ]:
import requests

def load_lora(lora_name: str, lora_path: str, port:int = 8000):
    url = f"http://localhost:{port}/v1/load_lora_adapter"
    headers = {"Content-Type": "application/json"}
    payload = {
        "lora_name": lora_name,
        "lora_path": lora_path,
    }

    try:
        response = requests.post(url, headers=headers, json=payload)
        response.raise_for_status()  # Raise an exception for HTTP errors (4xx or 5xx)

        print("Request successful!")
        print("Status Code:", response.status_code)
        # print("Response JSON:", response.json())
        return {"success": True, "status_code": response.status_code}
    except requests.exceptions.RequestException as e:
        print(f"Error making request: {e}")
        if hasattr(e, 'response') and e.response is not None:
            print("Response Status Code:", e.response.status_code)
            print("Response Text:", e.response.text)
        return {"success": False, "error": str(e)}


def unload_lora(lora_name:str, port:int = 8000):
    url = f"http://localhost:{port}/v1/unload_lora_adapter"
    headers = {"Content-Type": "application/json"}
    payload = {
        "lora_name": lora_name,
    }

    try:
        response = requests.post(url, headers=headers, json=payload)
        response.raise_for_status()  # Raise an exception for HTTP errors (4xx or 5xx)

        print("Request successful!")
        print("Status Code:", response.status_code)
        # print("Response JSON:", response.json())
        return {"success": True, "status_code": response.status_code}
    except requests.exceptions.RequestException as e:
        print(f"Error making request: {e}")
        if hasattr(e, 'response') and e.response is not None:
            print("Response Status Code:", e.response.status_code)
            print("Response Text:", e.response.text)
        return {"success": False, "error": str(e)}


In [ ]:
# use vllm inference client
# port = 8200
# client = AsyncOpenAI(base_url=f"http://localhost:{port}/v1", api_key="fake-key")
# client = patch_openai(client)

# use the art  client
port = 8000
client = model.openai_client()

In [ ]:
from art.utils.output_dirs import get_output_dir_from_model_properties, get_step_checkpoint_dir 

epochs = 20
n_rollouts = 600


prev_step_checkpoint_dir = None
step_checkpoint_dir = None
inference_name = model_name


for i in range(await model.get_step(), epochs):
    print(f"Starting step {i} for model {model.name}")
    
    #NOTE: If we use our own VLLM server, we need to uncomment the load and unload lora functions below
    # output_dir = get_output_dir_from_model_properties(project_name, name=run_name, art_path="./art")
    # prev_step_checkpoint_dir = step_checkpoint_dir
    # step_checkpoint_dir = get_step_checkpoint_dir(output_dir, i) if i > 0 else None
    # step_checkpoint_dir = step_checkpoint_dir.replace("./art/", ".art/") if step_checkpoint_dir is not None else None

    # if prev_step_checkpoint_dir is not None:
    #     print(f"Unloading lora")
    #     unload_lora(run_name, port)
    # if step_checkpoint_dir is not None:
    #     print(f"Loading lora from {step_checkpoint_dir}")
    #     res = load_lora(run_name, step_checkpoint_dir, port)
    #     if res["success"]:
    #         inference_name = run_name
    
    train_groups = await art.gather_trajectory_groups(
        (
            art.TrajectoryGroup(
                rollout(model, TicTacToeScenario(step=i), inference_name, client) for _ in range(n_rollouts)
            )
            for _ in range(1)
        ),
        pbar_desc="gather",
    )
    await model.delete_checkpoints()
    await model.train(train_groups, config=art.TrainConfig(learning_rate=5e-5))

In [ ]:
mean_chat_time = np.mean(timings["total_chat_time"])
mean_rollout_time = np.mean(timings["total_rollout_time"])
print(f"Mean chat time: {mean_chat_time:.2f} seconds")
print(f"Mean rollout time: {mean_rollout_time:.2f} seconds")
print(f"Total chat time: {sum(timings['total_chat_time']):.2f} seconds")
print(f"Total rollout time: {sum(timings['total_rollout_time']):.2f} seconds")


### Using the Model

Just like that, you've trained an agent to play tic tac toe! Now it's time to use your model outside of ART, in the wild! The easiest way to do that is to load it from disk, where it was saved after each training step, and either run inference on it locally or upload it to a central hub like HuggingFace.

Check out the code below for small demo of the model you just trained playing tic tac toe!


In [ ]:
import torch
from unsloth import FastLanguageModel


# example: .art/tic-tac-toe/models/002/0003
lora_model_path = (
    f".art/{model.project}/models/{model.name}/{await model.get_step():04d}"
)

print(f"loading model from {lora_model_path}\n")

peft_model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=lora_model_path,
    max_seq_length=16384,
    dtype=torch.bfloat16,
    load_in_4bit=True,
)
FastLanguageModel.for_inference(peft_model)

game = generate_game()
move_number = 0

messages = [
    {
        "role": "system",
        "content": f"You are a tic-tac-toe player. You are playing against an opponent. Always choose the move most likely to lead to an eventual win. Return your move as an XML object with a single property 'move', like so: <move>A1</move>. Optional moves are 'A1', 'B3', 'C2', etc. You are the {game['agent_symbol']} symbol.",
    },
]

if game["agent_symbol"] == "o":
    starting_opponent_move = get_opponent_move(game)
    game["board"][starting_opponent_move[0]][starting_opponent_move[1]] = game[
        "opponent_symbol"
    ]

while check_winner(game["board"]) is None:
    rendered_board = render_board(game)
    messages.append({"role": "user", "content": rendered_board})

    inputs = tokenizer.apply_chat_template(
        messages, return_tensors="pt", add_generation_prompt=True
    ).to("cuda")

    content = ""

    def get_completion() -> str:
        with torch.no_grad():
            outputs = peft_model.generate(
                input_ids=inputs,
                max_new_tokens=100,
                do_sample=True,
                temperature=0.7,
                top_p=0.9,
            )
            return tokenizer.decode(
                outputs[0][inputs.shape[1] :], skip_special_tokens=True
            )

    try:
        content = get_completion()
    except Exception as e:
        print("caught exception generating chat completion", e)
        raise e

    messages.append({"role": "assistant", "content": content})

    try:
        apply_agent_move(game, content)
        move_number += 1
    except ValueError:
        raise ValueError(f"Invalid move on move {move_number}: {content}")

    # print the board every move
    print(f"\nmove {move_number}")
    print(f"board:\n{rendered_board}")
    print(f"agent move: {content}")
    print(f"updated board:\n{render_board(game)}")

    if check_winner(game["board"]) is not None:
        break
    move_number += 1

    opponent_move = get_opponent_move(game)
    game["board"][opponent_move[0]][opponent_move[1]] = game["opponent_symbol"]

winner = check_winner(game["board"])

print(f"game finished in {move_number} moves")

if winner == game["agent_symbol"]:
    print("game won! 💪")
elif winner == game["opponent_symbol"]:
    print("game lost! 😢")
elif winner == "draw":
    print("draw! 🤷‍♂️")


print(f"final board:\n\n{render_board(game)}")


<div class="align-center">
<a href="https://github.com/openpipe/art"><img src="https://github.com/openpipe/art/raw/notebooks/assets/ART_pill.png" height="50"></a>
<a href="https://discord.gg/zbBHRUpwf4"><img src="https://github.com/openpipe/art/raw/notebooks/assets/Discord_pill.png" height="50"></a>
<a href="https://openpipe.ai/blog/art-e-mail-agent"><img src="https://github.com/openpipe/art/raw/main/assets/ART_E_pill.png" height="50"></a>

Questions? Join the Discord and ask away! For feature requests or to leave a star, visit our [Github](https://github.com/openpipe/art).

</div>
